In [1]:
from source import helpers

import pandas as pd
import numpy as np
import regex as re
import json
import os

    

class lpa_prep():
    def __init__(self,
                 metric : str):
        """
        metric : str -> Metric on which the analysis will be performed.
        substitution_survival - analyze the selection patterns (through non-syn mutations survival) across the heavy variable regions of the immune reportiore.
        trimers_usage - analyze the trimers (k-mers) usage of across the heavy variable regions of the immune reportiore.
        """
        
        if metric not in ["substitution_survival", "trimers_usage"]:
            raise Exception("Invalid metric input, valid inputs are: 'substitution_survival', 'trimers_usage'.")

        # Required tables for the selected analysis
        req_tables = {"substitution_survival":["clones", "clone_stats", "sample_metadata"],
                      "trimers_usage":[]}
        
        # Defining varibles to be used across the analysis
        self.config = helpers.read_json() #config information, used for sql connection
        self.metric = metric # metric on which the data will be analyzed
        self.req_tables = req_tables[metric] # required sql tables
        self.analysis_mame = "-".join([self.config["sql"]["database"], self.metric]) # analysis name 

        # required folders per step
        self.paths = {"data_imports": os.path.join("data_imports", self.analysis_mame),
                      "data_processed": os.path.join("data_processed", self.analysis_mame),
                      "lpa_input": os.path.join("lpa_input", self.analysis_mame),
                      "lpa_results": os.path.join("lpa_results", self.analysis_mame)}
        
        helpers.create_folders([os.path.join(i, self.analysis_mame) for i in self.paths])

    # Class methods which imports raw tables from the sql server
    def import_tables(self):
        sql_database = self.config["sql"]["database"]
        data_path = self.paths["data_imports"]
        imported_tabeles = []

        try:
            sql_connecntion = helpers.mysql_qry()
            print(f"> Connected to the MySQL server ({sql_database}).")

        except:
            print(f"> Failed to connect to the MySQL server ({sql_database}).")
        
        for i in self.req_tables:
            print(f"{i}.csv:")
            i_path = os.path.join(data_path, i+".csv")

            if os.path.exists(i_path):
                print(f"> raw table `{i}` already exists at {i_path}. continuing.")

            else:
                print(f"> importing table `{i}` to {i_path}.")

                temp_qry = f"""
                            SELECT * FROM {sql_database}.{i};
                            """
              
                # Incase of invalid input error
                try:
                    temp_df = sql_connecntion.run_qry(temp_qry)
                    temp_df.to_csv(i_path)
                    imported_tabeles.append(i)

                except:
                    print(f"> Invalid database or table name. (db={self.config["sql"]["database"]}, table={i})")
                    
        sql_connecntion.close_conn()

        files_actual = np.sort(os.listdir(data_path))
        files_expected = np.sort([i+".csv" for i in self.req_tables])
        print(f"imported tabled:{files_actual} from required talbed: {files_expected} ({len(files_actual)}/{len(files_expected)})")

    def process_tables(self):
        if self.metric == "substitution_survival":            
            """
            * Creating orginized metadata dataframe with the information provided by the config.json file.
            * Saving the metadata_df into folder.
            * If the dataframe already exists, load it without processing.
            """
            path_processed_dir = self.paths["data_processed"]
            path_metdadata_df = os.path.join(path_processed_dir, "sample_metadata_df.csv")

            if os.path.exists(path_metdadata_df):
                print("> sample_metadata_df.csv already created, continuing...")
                metadata_df = pd.read_csv(path_metdadata_df, index_col=0)
                                                                                        
            else:
                print("> Creating sample_metadata_df.csv.")
            
                metadata_keys_og = self.config["sql"]["metadata_columns"].split(",")
                metadata_keys_new =  self.config["sql"]["rename_metdata_columns"].split(",")
                meta_dict = dict(zip(metadata_keys_og, metadata_keys_new))

                metadata_df = pd.read_csv(os.path.join(self.paths["data_imports"], "sample_metadata.csv"), index_col=0)
                metadata_og = metadata_df[metadata_df["key"].isin(metadata_keys_og)]
                metadata_og = metadata_og.replace({"key":meta_dict})

                sample_ids = np.sort(metadata_og["sample_id"].unique())
                metadata_df = pd.DataFrame({"sample_id":sample_ids})
                metadata_df[metadata_keys_new] = np.nan

                for i in sample_ids:
                    temp_sid = i
                    for j in metadata_keys_new:
                        cond_sid = (metadata_og["sample_id"] == i)
                        cond_key = (metadata_og["key"] == j)
                        metadata_df.loc[metadata_df["sample_id"]==i,j] = metadata_og.loc[(metadata_og["sample_id"]==i)&(metadata_og["key"]==j),"value"].values
                metadata_df.to_csv(path_metdadata_df)
                print("> Done.")

            # Creation of filtred metadata table
            """
            * Creating custom function to pull metadata from metadata_df
            * sample_id validation
            """
            def assign_metadata(sample_id, meta_df):
                meta_list = meta_df.columns[1:]
                meta_sample = meta_df.loc[meta_df["sample_id"]==sample_id, meta_list].values.flatten()
                return meta_sample
            
            clone_stats = pd.read_csv(os.path.join(self.paths["data_imports"], "clone_stats.csv"), index_col=0)
            metalist_sids = np.sort(metadata_df.sample_id.unique())
            clones_sids = np.sort(clone_stats.dropna().sample_id.unique()).astype("int")

            values_missing = np.setdiff1d(clones_sids, metalist_sids)
            values_common = np.intersect1d(metalist_sids, clones_sids)

            if len(values_missing) > 0:
                print("> missing sample_id from metadata file:",values_missing)
                clone_stats = clone_stats[clone_stats["sample_id"].isin(values_common)]
                raise TypeError("verify metadata sample_id values") 
        
            # Merging clones and clones status > adding the relevent metadata to the dataframe
            """
            * loading clones_merged if exists, if not creating and saving
            * custom function that extract mutations infromation from the "mutation" json in each row
            * Adding the germline infromation to the clone_stats df
            * Dropping null sample_id rows (cannot assign metadata for those rows)
            * converting "sample_id" values to int instead of floats
            * assiging the metadata into the merged table (applying assign_metadata)
            * renaming id_x to id after merging (left had "id" colum while right had "id"=="clone_id")
            * reseting the index
            """
            
            path_clones_merged = os.path.join(self.paths["data_processed"], "clones_merged.csv")

            if os.path.exists(path_clones_merged):
                clones_merged = pd.read_csv(path_clones_merged)
                print("> clones_merged dataframe exists, loading and continuing....")

            else: 
                clones = pd.read_csv(os.path.join(self.paths["data_imports"], "clones.csv"))
                clones_merged = clone_stats.merge(right=clones[["id","germline"]],
                                                    how="left",
                                                    left_on="clone_id",
                                                    right_on="id")    
                
                clones_merged = clones_merged[clones_merged["sample_id"].notnull()]        
                clones_merged[list(metadata_df.columns)[1:]] = list(clones_merged["sample_id"].apply(assign_metadata, args=(metadata_df,)))
                clones_merged.rename({"id_x":"id"},axis="columns",inplace=True)
                clones_merged.reset_index(drop=True, inplace=True)
                clones_merged.to_csv(path_clones_merged)
                print("> clones_merged dataframe created and saved, continuing....")

            # Creating the mutation dataframe
            """
            * Creating df with the relevent mutations infromation for each clone (mut_df)
            * Cleaning the mut_df and adding relevent data
            * Saving the mut_df
            """

            path_mut_df = os.path.join(self.paths["data_processed"], "mut_df.csv")

            if os.path.exists(path_mut_df):
                mut_df = pd.read_csv(path_mut_df,index_col=0)
                print("> mut_df dataframe exists, loading and continuing....")

            else: 
                def mut_regall(string):
                    pattern = r"'pos': (?P<position>\d+), 'from_nt': '(?P<from_nt>[\w]+)', 'from_aa': '(?P<from_aa>[\w\*]+)', 'to_nt': '(?P<to_nt>['\w\*]+)', 'to_aas': \[(?P<to_aas>['\w,\s\*]+)], 'unique': (?P<unique>\d+), 'total': (?P<total>\d+), 'intermediate_aa': '(?P<intermediate_aa>[\w\d\*])'"
                
                    tjson = json.loads(string)
                    
                    if "ALL" in str(tjson["regions"].keys()):
                        all_value = str(tjson["regions"]["ALL"])
                        find = re.findall(pattern,all_value)
                        return find
                    
                    else:
                        else_value = str(tjson["regions"])
                        return else_value
                        
                clones_merged["regions_all"] = clones_merged["mutations"].apply(mut_regall)
                clones_raval = clones_merged.copy()
                ra_val = []
                
                for i in range(0,len(clones_raval)):
                    length = len(clones_raval.loc[i,"regions_all"]) # length of the list, number of mutations is the colum
                    value = clones_raval.loc[i,"regions_all"] # the value mutations themselves list of lists/ list / np.nan
                    id_value = clones_raval.loc[i,"id"] # id value of the row
                    clone_id = clones_raval.loc[i,"clone_id"] # clone_id value of the row
                    subject_id = clones_raval.loc[i,"subject_id"]# subject_id value of the row
                    sample_id = clones_raval.loc[i,"sample_id"] # sample_id value of the row
                    funct = clones_raval.loc[i,"functional"] # functional value of the clone
                    total_cnt = clones_raval.loc[i,"total_cnt"] # target of the antibody
                    unique_cnt = clones_raval.loc[i,"unique_cnt"] # unique sequence is clone
                    germline = clones_raval.loc[i,"germline"] #germline sequence
                    top_seq = clones_raval.loc[i,"top_copy_seq_sequence"] #top copy of sequence
                    metadata = clones_raval.loc[i,metadata_df.columns[1:]].values.flatten().tolist() #metadata list value
                    ins_val = [id_value, clone_id, subject_id, sample_id, funct, total_cnt,unique_cnt, germline, top_seq] + metadata
                    
                    # if single row of mutation
                    if length == 1:
                        to_aas = value[0][4].replace(" ","").replace("''","").split(",")
                        
                        if (len(to_aas) == 1):
                            temp_list = list(value[0])
                            ra_val.append(ins_val + temp_list) 
                            
                        else:
                            for i in range(0,len(to_aas)):
                                temp_list = list(value[0])
                                temp_list[4] = to_aas[i]
                                ra_val.append(ins_val + temp_list)
                    
                    # if multiple rows of mutations
                    if length > 1:
                        for j in range(0,length):
                                sub_value = list(value[j]) #each row
                                temp_list = sub_value
                                
                                # making sure that the length of the list is correct, in some rows there is missing values
                                if len(sub_value) == 8:
                                    to_aas = sub_value[4].replace(" ","").replace("'","").split(",")
                                    
                                    if len(to_aas) == 1:
                                        ra_val.append(ins_val + temp_list)
                                    elif len(to_aas) > 1:
                                        for aa in set(to_aas): # set() removes duplicate values
                                            temp_list[4] = aa
                                            ra_val.append(ins_val + temp_list)
                                                    
                    elif length == 0:
                        ra_val.append(ins_val + np.full(shape=len(value), fill_value=np.nan).tolist())
                
                mut_df_cols = ["id", "clone_id", "subject_id", "sample_id", "functional", "total_cnt","unique_cnt", "germline", "top_seq"]
                mut_info_cols = ["pos","from_nt","from_aa","to_nt","to_aas","unique","total","intermidiate_aa"]
                
                mut_df = pd.DataFrame(data=ra_val, columns = mut_df_cols + metadata_df.columns[1:].tolist() + mut_info_cols)
                mut_df["to_aas"] = mut_df["to_aas"].str.replace("'","") #cleaning to_aas string
                mut_df.replace({"to_aas":{"None":np.nan}}, inplace=True) #turining none values to np.nan
                mut_df.dropna(axis=0,subset=["pos","to_aas"], ignore_index=True, inplace=True) #dropping null rows of "pos" and "to_aas"

                # custom function to round numbers upward
                def round_up(number):
                    num_dec = number
                    num_round = round(number)
                    
                    if num_round < num_dec:
                        value = num_round + 1
                    else:
                        value = num_round
                    return value
                
                mut_df.insert(6,"pos_aa",np.nan) #inserting amino acid position column
                mut_df.insert(6,"pos_nt",np.nan) #inserting nucleotide position column
                mut_df.loc[:,"pos_nt"] = mut_df.loc[:,"pos"].apply(int)+1 #filling the pos_nt column
                mut_df.loc[:,"pos_aa"] = ((mut_df.loc[:,"pos_nt"])/3).apply(round_up) #fillint the pos_aa column 
                mut_df.astype({"pos_nt":"int","pos_aa":"int"})
                mut_df.drop(axis=1,columns="pos",inplace=True) #dropping the og column (it was -1 in position...)
                mut_df["syn"] = (mut_df["from_aa"] == mut_df["to_aas"]).apply(int) #creating syn column

                mut_df.to_csv(path_mut_df)
                print("> mut_df dataframe created and saved, continuing....")
                
            print("> Successfully created mutation dataframe (mut_df.csv)")


        elif self.metric == "trimers_usage":
            pass

    def create_docuemnts(self):
        pass


> Dir `data_imports` already exists.
> Dir `data_processed` already exists.
> Dir `lpa_input` already exists.
> Dir `lpa_results` already exists.


In [2]:
# Creating the LPA preprocessing object, defining the analysis metric.
lpa_preprocessing = lpa_prep(metric = "substitution_survival")

> Dir `data_imports\covid_vaccine_new-substitution_survival` already exists.
> Dir `data_processed\covid_vaccine_new-substitution_survival` already exists.
> Dir `lpa_input\covid_vaccine_new-substitution_survival` already exists.
> Dir `lpa_results\covid_vaccine_new-substitution_survival` already exists.


In [3]:
# Importing the data from the mysql server (if required).
lpa_preprocessing.import_tables()

> Established connecntion to the covid_vaccine_new database.
> Connected to the MySQL server (covid_vaccine_new).
clones.csv:
> raw table `clones` already exists at data_imports\covid_vaccine_new-substitution_survival\clones.csv. continuing.
clone_stats.csv:
> raw table `clone_stats` already exists at data_imports\covid_vaccine_new-substitution_survival\clone_stats.csv. continuing.
sample_metadata.csv:
> raw table `sample_metadata` already exists at data_imports\covid_vaccine_new-substitution_survival\sample_metadata.csv. continuing.
> MySQL connenction terminated.
imported tabled:['clone_stats.csv' 'clones.csv' 'sample_metadata.csv'] from required talbed: ['clone_stats.csv' 'clones.csv' 'sample_metadata.csv'] (3/3)


In [4]:
lpa_preprocessing.process_tables()

> sample_metadata_df.csv already created, continuing...
> clones_merged dataframe exists, loading and continuing....
> mut_df dataframe created and saved, continuing....
> Successfully created mutation dataframe (mut_df.csv)
